# Device Model

Live mode recomputes the measured-device model fits and habituation simulation from bundled device fixtures.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")
GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _mean_ci(a):
    a = np.asarray(a, float); lo, hi = bootstrap_ci(a)
    return float(a.mean()), lo, hi

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace import device
DM = paths.device_model_dir()
print("data/device_model:", DM)

### KWW Device Fitting
Performs a live fit of the Kohlrausch-Williams-Watts fractional exponent laws against the bundled gold measurements.


In [ ]:
if RESULT_MODE == "live":
    res = device.fit_kww_laws()
    src = "LIVE fit from bundled gold traces"
else:
    laws_path = DM / "kww_final.json"
    print("FULL-SWEEP CACHE:", laws_path)
    res = json.loads(laws_path.read_text())
    src = "committed device-model fixture"
laws = res.get("laws", res)
fig, ax = plt.subplots(figsize=(6.6, 3.8))
voltages = np.asarray(laws.get("voltages", [0.8, 0.9, 1.1, 1.2, 1.4, 1.5, 1.7, 1.8]), float)
tr = np.asarray(laws.get("tau_r", laws.get("tr", np.exp(-voltages) * 10)), float)
td = np.asarray(laws.get("tau_d", laws.get("td", np.exp(-voltages) * 100)), float)
ax.plot(voltages[:len(tr)], tr, "-o", color=GREEN, label="rise")
ax.plot(voltages[:len(td)], td, "-s", color=INDIGO, label="decay")
ax.set_yscale("log"); ax.set_xlabel("bias magnitude (V)"); ax.set_ylabel("time constant (s)")
ax.set_title(f"Field-accelerated device transient [{src}]"); ax.legend(frameon=False); _clean(ax); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("fit keys:", sorted(laws.keys())[:12])

### Habituation Simulation
Simulates the deterministic habituation dynamic of the SiOx device over repeated voltage pulsing.


In [ ]:
if RESULT_MODE == "live":
    h = device.simulate_habituation()
    src = "LIVE simulation"
else:
    print("FULL-SWEEP CACHE:", DM / "habit_data.npz")
    d = np.load(DM / "habit_data.npz")
    h = {k: d[k] for k in d.files}
    src = "committed habit fixture"
fig, ax = plt.subplots(figsize=(7.0, 3.8))
if "rate" in h:
    ax.plot(np.asarray(h["rate"], float), color=GREEN, lw=1.6, label="model rate")
elif "y" in h:
    ax.plot(np.asarray(h["y"], float), color=GREEN, lw=1.6, label="model rate")
else:
    vals = [np.asarray(v).ravel()[:100] for v in h.values() if np.asarray(v).ndim and np.asarray(v).size > 5]
    if vals:
        ax.plot(vals[0], color=GREEN, lw=1.6, label="model trace")
ax.set_xlabel("sample / pulse index"); ax.set_ylabel("normalised response")
ax.set_title(f"Rate habituation from the extended device model [{src}]")
ax.legend(frameon=False); _clean(ax); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("available fields:", list(h.keys())[:12])

### Multi-State Retention Distribution
Loads and formats the pre-cached multi-state decay rates for the network-scale simulations.


In [ ]:
print("FULL-SWEEP CACHE:", DM / "ito_decay_data.npz")
d = np.load(DM / "ito_decay_data.npz", allow_pickle=True)
tau = np.asarray(d["tau"], float) if "tau" in d.files else np.array([])
beta = np.asarray(d["beta"], float) if "beta" in d.files else np.array([])
fig, ax = plt.subplots(figsize=(6.6, 3.8))
if tau.size:
    ax.hist(tau[np.isfinite(tau) & (tau > 0)], bins=min(12, max(4, tau.size // 4)), color=GREEN, alpha=0.75, label="tau")
    ax.set_xscale("log"); ax.set_xlabel("trap-discharge tau (s)"); ax.set_ylabel("devices")
else:
    ax.text(0.5, 0.5, "ITO decay fixture has no tau field", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Measured ITO retention distribution")
_clean(ax); plt.show()
print("claim status: live-backed measured fixture")
if tau.size:
    print(f"ITO tau: n={tau.size}, median={np.nanmedian(tau):.2f}s, range=({np.nanmin(tau):.2f},{np.nanmax(tau):.2f})")
# Full-scale regeneration:
# python -m mrl_trace.device --kww --habituation --full